# boolean-mask-identity-replace — ex5: safe batched solve (singular → zero solution)

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `boolean-mask-identity-replace`. When a test cell passes, your progress is reported back to your account.

**What you'll practice.** Five mask-and-substitute patterns that ramp from `x < 0` → clamp → row-zero → identity-substitute → safe batched solve. Read the docstring, fill the function body, run the test cell. The solution sits in the collapsed `<details>` block below each exercise.

**Per-exercise structure** (Doughty et al. ACE 2024 — `[Bloom level] + [LO] + [Keywords] + [KCs]`):
Each exercise begins with a yaml block stating its Bloom cognitive level, learning objective, keywords, and the knowledge components (KCs) it targets. This makes the cognitive demand explicit instead of buried.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Numpy: Indexing and selection` subtopic.
You can copy the token from your Delta Drills account page.

This drill exercises the **atom `boolean-mask-identity-replace`**, which bridges to the bank subtopic `Numpy: Indexing and selection` for EWMA state. Completing all 5 exercises triggers a single `arena-rating` beacon at the end of the notebook.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "boolean-mask-identity-replace"
DD_SUBTOPIC = "Numpy: Indexing and selection"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

# Track which exercises passed in this session.
_dd_passed = set()

## Mask & substitute — quick refresher

**Build a mask.** Any comparison returns a `dtype=bool` tensor with the same shape as its input: `x < 0`, `x.abs() < eps`, `(x > 0) & (x < 1)`.

**Write through a mask.** `y[mask] = value` modifies in place. If `value` is a scalar, it broadcasts over the masked region. If `value` is a tensor, its shape must match the shape of `y[mask]` after broadcasting.

**Always clone first** if you want a non-mutating function — `y = x.clone(); y[mask] = 0; return y`. Otherwise the caller's input gets clobbered.

**Identity substitute.** `A[singular_mask] = torch.eye(N)` replaces flagged `(N, N)` submatrices with the identity — the standard cleanup before a batched `linalg.solve` so one degenerate slice doesn't crash the whole batch.

### Exercise 5 — safe batched solve (singular → zero solution)

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Create
> LO: Synthesize singularity detection, identity substitution in `A`, zero substitution in `b`, and a batched solve to produce a per-slot solution where degenerate slots come out as zero vectors.
> Keywords: batched-solve, det, ray-triangle, multi-kc
> ```

**KCs targeted:** `mask-from-condition`, `identity-substitute-singular-batched`, `row-mask-broadcast-assign`, `safe-batched-solve`

Implement `ex5_safe_solve(A, b, eps=1e-6)`. Solve a batched linear system `A @ x = b` where some `A` slices may be singular.

Inputs:
- `A`: `(B, N, N)` float tensor.
- `b`: `(B, N)` float tensor.
- `eps`: singularity threshold on `|det(A_i)|`.

Algorithm:
1. Compute `dets = torch.det(A)` → shape `(B,)`.
2. `singular_mask = dets.abs() < eps` — flags degenerate slots.
3. `A_safe = A.clone(); A_safe[singular_mask] = eye(N, dtype=A.dtype)`.
4. `b_safe = b.clone(); b_safe[singular_mask] = 0.0`.
5. Solve `torch.linalg.solve(A_safe, b_safe.unsqueeze(-1)).squeeze(-1)`.
6. Return result of shape `(B, N)`. Degenerate slots come out as zeros.

Neither `A` nor `b` should be mutated.

> ⚠️ **Integrative exercise.** Combines 4 KCs (mask-from-condition, identity-substitute-batched, row-mask-broadcast-assign, safe-batched-solve). Empirical work (Lohr et al. ITiCSE 2025) shows 3+ concept exercises drop to ~40% solvability — expect a clear step up vs Exercises 1-4.

In [ ]:
def ex5_safe_solve(A: Tensor, b: Tensor, eps: float = 1e-6) -> Tensor:
    """Batched solve with singular-A substitution. Degenerate slots → zero vector."""
    raise NotImplementedError()


def _test_ex5():
    # Two well-conditioned systems and one singular.
    A = t.tensor([
        [[2.0, 0.0], [0.0, 1.0]],   # solve: x = b / [2, 1]
        [[0.0, 0.0], [0.0, 0.0]],   # singular — det = 0
        [[1.0, 1.0], [0.0, 1.0]],   # upper triangular, det = 1
    ])
    b = t.tensor([
        [4.0, 3.0],   # → [2.0, 3.0]
        [5.0, 6.0],   # singular slot — must come out [0, 0]
        [3.0, 2.0],   # → [1.0, 2.0]
    ])
    A_orig = A.clone(); b_orig = b.clone()
    out = ex5_safe_solve(A, b)
    assert out.shape == b.shape, f'expected {b.shape}, got {out.shape}'
    # Well-conditioned slots: actual linear-system solutions.
    assert t.allclose(out[0], t.tensor([2.0, 3.0]), atol=1e-5), f'slot 0 wrong: {out[0]}'
    assert t.allclose(out[2], t.tensor([1.0, 2.0]), atol=1e-5), f'slot 2 wrong: {out[2]}'
    # Singular slot: zero vector (not NaN, not error).
    assert t.allclose(out[1], t.zeros(2)), f'singular slot must be zero, got {out[1]}'
    assert not t.isnan(out).any(), 'output must contain no NaN'
    # Non-mutation.
    assert t.allclose(A, A_orig), 'A was mutated'
    assert t.allclose(b, b_orig), 'b was mutated'
    # No-singular case — output must match a plain solve.
    A2 = t.tensor([[[3.0, 0.0], [0.0, 2.0]], [[1.0, 0.0], [0.0, 1.0]]])
    b2 = t.tensor([[6.0, 4.0], [5.0, 7.0]])
    out2 = ex5_safe_solve(A2, b2)
    expected2 = t.linalg.solve(A2, b2.unsqueeze(-1)).squeeze(-1)
    assert t.allclose(out2, expected2, atol=1e-5), 'no-singular case must match plain solve'
    _dd_passed.add('ex5')
    print("ex5 ✓")

_test_ex5()

<details><summary>Solution</summary>

```python
def ex5_safe_solve(A: Tensor, b: Tensor, eps: float = 1e-6) -> Tensor:
    dets = t.det(A)
    singular = dets.abs() < eps
    A_safe = A.clone()
    b_safe = b.clone()
    N = A.shape[-1]
    A_safe[singular] = t.eye(N, dtype=A.dtype)
    b_safe[singular] = 0.0
    sol = t.linalg.solve(A_safe, b_safe.unsqueeze(-1)).squeeze(-1)
    return sol
```

**Why this matters in Ray Tracing.** Ray-triangle intersection boils down to a 3×3 linear system per ray (the Möller-Trumbore matrix). Rays parallel to a triangle's plane give a singular matrix — and if you call `torch.linalg.solve` on the batch, ONE singular system crashes the WHOLE call. Substituting identity in `A` and zero in `b` lets the batched solve complete; the originally-singular slots come out as zeros, which the caller treats as 'no intersection'.

**Why solve on `b.unsqueeze(-1)` not `b` directly?** `torch.linalg.solve(A, b)` accepts both `(B, N)` and `(B, N, K)` for `b`. The `.unsqueeze(-1)` form is explicit and consistent across PyTorch versions; we squeeze the trailing 1 to get back to `(B, N)`.

**Why `eps = 1e-6` and not `1e-12`?** Determinants of `N×N` matrices scale with the magnitude of entries — a near-singular but numerically small `det` can mean a poorly-conditioned, not actually-singular, matrix. `1e-6` is a conservative threshold for float32 work; tune based on your expected input magnitudes.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex5'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex5',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',  # single-exercise standalone — neutral signal
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()